In [ ]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 9.9 MB/s  0:00:00


In [ ]:
import pandas as pd
import random
from faker import Faker

fake = Faker()
random.seed(42)

# Configuration
NUM_NORMAL_USERS = 500
NUM_BOTS = 50
NUM_POSTS = 200

print("Generating users...")
users = []
user_ids = []

# 1. Generate Normal Users
for i in range(NUM_NORMAL_USERS):
    uid = f"user_{i}"
    user_ids.append(uid)
    users.append({
        "userId": uid,
        "name": fake.name(),
        "isBot": False,
        "createdAt": fake.date_time_between(start_year=2023, end_year=2025).strftime("%Y-%m-%d")
    })

# 2. Generate Bot Ring Users
bot_ids = []
for i in range(NUM_BOTS):
    uid = f"bot_{i}"
    bot_ids.append(uid)
    users.append({
        "userId": uid,
        "name": f"Bot_{fake.user_name()}",
        "isBot": True,
        "createdAt": "2026-01-01"
    })

users_df = pd.DataFrame(users)
users_df.to_csv("users.csv", index=False)

print("Generating follows network...")
follows = []

# Normal users follow randomly (Power-law style approximation)
for uid in user_ids:
    # Each normal user follows 5 to 30 random people
    followed = random.sample(user_ids, k=random.randint(5, 30))
    for f in followed:
        if f != uid:
            follows.append({"source": uid, "target": f, "type": "ORGANIC"})

# Plant Bot Ring 1: Reciprocal Follow Ring (A -> B -> C -> A close loop)
for i in range(len(bot_ids)):
    source = bot_ids[i]
    target = bot_ids[(i + 1) % len(bot_ids)] # points to the next bot in circle
    follows.append({"source": source, "target": target, "type": "BOT_RING"})

    # Add a few cross-follows within bot group to make it a dense clique
    for _ in range(3):
        rand_bot = random.choice(bot_ids)
        if rand_bot != source:
            follows.append({"source": source, "target": rand_bot, "type": "BOT_RING"})

follows_df = pd.DataFrame(follows).drop_duplicates()
follows_df.to_csv("follows.csv", index=False)

print("Generating posts and likes...")
posts = []
likes = []

for i in range(NUM_POSTS):
    pid = f"post_{i}"
    author = random.choice(user_ids + bot_ids)
    posts.append({"postId": pid, "authorId": author, "timestamp": fake.iso8601()})

posts_df = pd.DataFrame(posts)
posts_df.to_csv("posts.csv", index=False)

# Plant Co-Amplification Hub: All bots like the same set of specific target posts instantly
target_bot_posts = [p['postId'] for p in posts if p['authorId'] in bot_ids[:5]]

for uid in user_ids:
    # Normal users like random posts
    liked_posts = random.sample([p['postId'] for p in posts], k=random.randint(1, 10))
    for pid in liked_posts:
        likes.append({"userId": uid, "postId": pid})

for b_id in bot_ids:
    # Bots aggressively co-amplify the target posts
    for pid in target_bot_posts:
        likes.append({"userId": b_id, "postId": pid})

likes_df = pd.DataFrame(likes).drop_duplicates()
likes_df.to_csv("likes.csv", index=False)

print("Dataset generation complete! Created: users.csv, follows.csv, posts.csv, likes.csv")

Generating users...


TypeError: Provider.date_time_between() got an unexpected keyword argument 'start_year'

In [ ]:
import pandas as pd
import random
from faker import Faker

fake = Faker()
random.seed(42)

# Configuration
NUM_NORMAL_USERS = 500
NUM_BOTS = 50
NUM_POSTS = 200

print("Generating users...")
users = []
user_ids = []

# 1. Generate Normal Users
for i in range(NUM_NORMAL_USERS):
    uid = f"user_{i}"
    user_ids.append(uid)
    users.append({
        "userId": uid,
        "name": fake.name(),
        "isBot": False,
        "createdAt": fake.date_between(start_date="-3y", end_date="today").strftime("%Y-%m-%d")
    })

# 2. Generate Bot Ring Users
bot_ids = []
for i in range(NUM_BOTS):
    uid = f"bot_{i}"
    bot_ids.append(uid)
    users.append({
        "userId": uid,
        "name": f"Bot_{fake.user_name()}",
        "isBot": True,
        "createdAt": "2026-01-01"
    })

users_df = pd.DataFrame(users)
users_df.to_csv("users.csv", index=False)

print("Generating follows network...")
follows = []

# Normal users follow randomly
for uid in user_ids:
    followed = random.sample(user_ids, k=random.randint(5, 30))
    for f in followed:
        if f != uid:
            follows.append({"source": uid, "target": f, "type": "ORGANIC"})

# Plant Bot Ring 1: Reciprocal Follow Ring
for i in range(len(bot_ids)):
    source = bot_ids[i]
    target = bot_ids[(i + 1) % len(bot_ids)]
    follows.append({"source": source, "target": target, "type": "BOT_RING"})

    for _ in range(3):
        rand_bot = random.choice(bot_ids)
        if rand_bot != source:
            follows.append({"source": source, "target": rand_bot, "type": "BOT_RING"})

follows_df = pd.DataFrame(follows).drop_duplicates()
follows_df.to_csv("follows.csv", index=False)

print("Generating posts and likes...")
posts = []
likes = []

for i in range(NUM_POSTS):
    pid = f"post_{i}"
    author = random.choice(user_ids + bot_ids)
    posts.append({"postId": pid, "authorId": author, "timestamp": fake.iso8601()})

posts_df = pd.DataFrame(posts)
posts_df.to_csv("posts.csv", index=False)

target_bot_posts = [p['postId'] for p in posts if p['authorId'] in bot_ids[:5]]

for uid in user_ids:
    liked_posts = random.sample([p['postId'] for p in posts], k=random.randint(1, 10))
    for pid in liked_posts:
        likes.append({"userId": uid, "postId": pid})

for b_id in bot_ids:
    for pid in target_bot_posts:
        likes.append({"userId": b_id, "postId": pid})

likes_df = pd.DataFrame(likes).drop_duplicates()
likes_df.to_csv("likes.csv", index=False)

print("Dataset generation complete! Created: users.csv, follows.csv, posts.csv, likes.csv")

Generating users...
Generating follows network...
Generating posts and likes...
Dataset generation complete! Created: users.csv, follows.csv, posts.csv, likes.csv


In [ ]:
!pip install neo4j

In [ ]:
from neo4j import GraphDatabase
import pandas as pd

# Connection details directly from your downloaded file
URI = "neo4j+s://bd89ea29.databases.neo4j.io"
AUTH = ("bd89ea29", "cMWPdrsPt-YNdPO-qpJw-e-qBN4f7Rhf_ugPE77Liuk")

driver = GraphDatabase.driver(URI, auth=AUTH)

# Load the CSVs generated earlier from your Colab workspace
users_df = pd.read_csv("users.csv")
follows_df = pd.read_csv("follows.csv")
posts_df = pd.read_csv("posts.csv")
likes_df = pd.read_csv("likes.csv")

print("Uploading dataset to Neo4j AuraDB...")

with driver.session() as session:
    # 1. Create Users
    session.run("""
        UNWIND $rows AS row
        CREATE (:User {
            userId: row.userId,
            name: row.name,
            isBot: row.isBot,
            createdAt: row.createdAt
        })
    """, rows=users_df.to_dict(orient="records"))

    # 2. Create Constraints / Indexes for speed
    session.run("CREATE CONSTRAINT user_id_unique IF NOT EXISTS FOR (u:User) REQUIRE u.userId IS UNIQUE")
    session.run("CREATE CONSTRAINT post_id_unique IF NOT EXISTS FOR (p:Post) REQUIRE p.postId IS UNIQUE")

    # 3. Create Follows Relationships
    session.run("""
        UNWIND $rows AS row
        MATCH (u1:User {userId: row.source})
        MATCH (u2:User {userId: row.target})
        CREATE (u1)-[:FOLLOWS {type: row.type}]->(u2)
    """, rows=follows_df.to_dict(orient="records"))

    # 4. Create Posts
    session.run("""
        UNWIND $rows AS row
        MATCH (u:User {userId: row.authorId})
        CREATE (p:Post {postId: row.postId, timestamp: row.timestamp})
        CREATE (u)-[:POSTED]->(p)
    """, rows=posts_df.to_dict(orient="records"))

    # 5. Create Likes Relationships
    session.run("""
        UNWIND $rows AS row
        MATCH (u:User {userId: row.userId})
        MATCH (p:Post {postId: row.postId})
        CREATE (u)-[:LIKED]->(p)
    """, rows=likes_df.to_dict(orient="records"))

driver.close()
print("Successfully uploaded everything to Neo4j AuraDB! 🎉")

Uploading dataset to Neo4j AuraDB...


ConstraintError: {neo4j_code: Neo.ClientError.Schema.ConstraintValidationFailed} {message: Node(0) already exists with label `User` and property `userId` = 'user_0'} {gql_status: 22N79} {gql_status_description: error: data exception - property uniqueness constraint violated. Property uniqueness constraint violated: Node(0) already exists with label `User` and property `userId` = 'user_0'.}

In [1]:
import shutil

# Zip your CSV files
shutil.make_archive('project_datasets', 'zip', '.')

# Trigger the download in your browser
from google.colab import files
files.download('project_datasets.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>